# DATA-01 · EDA văn bản tin thật / tin giả

Bộ dữ liệu: [Fake and Real News Dataset](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset). Chạy toàn bộ sau khi đặt **file** `Fake.csv` và `True.csv` trong `ml/data/raw/`. Xem [hướng dẫn](README.md) để chọn kernel.

**Phạm vi:** mô tả nhãn *FAKE/REAL của bộ dữ liệu*, không xác minh sự thật. Notebook đọc văn bản thô để khảo sát dấu nguồn. Chỉ hiển thị bảng tổng hợp và biểu đồ; không in nội dung từng bài. Ảnh xuất ra `ml/artifacts/eda/` bị Git bỏ qua.

**Câu hỏi:** (1) Độ dài tiêu đề/nội dung khác nhau ra sao? (2) Chủ đề `subject` phân bố thế nào? (3) Từ nào hiện diện khác biệt theo nhãn? (4) Các chỉ số phong cách viết khác nhau ra sao? (5) Có dấu hiệu nguồn/định dạng mở đầu gây rò rỉ nhãn không?

In [ ]:
from pathlib import Path
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

def repo_root():
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / 'ml' / 'src' / 'train_baseline.py').is_file():
            return path
    raise FileNotFoundError('Mở notebook từ bên trong repository sic-fake-news-detector.')

ROOT = repo_root()
DATA_DIR = Path(os.environ.get('SIC_DATA_DIR', ROOT / 'ml' / 'data' / 'raw')).expanduser()
FIG_DIR = ROOT / 'ml' / 'artifacts' / 'eda'
FIG_DIR.mkdir(parents=True, exist_ok=True)
COLORS = {'FAKE': '#d95f5f', 'REAL': '#377eb8'}
pd.set_option('display.max_rows', 30)

def save_plot(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / name, dpi=150, bbox_inches='tight')
    plt.show()

print('Đọc dữ liệu từ:', DATA_DIR)
print('Biểu đồ lưu tại:', FIG_DIR)

## 1. Nạp dữ liệu và kiểm tra chất lượng

Không dùng `head()` trên dữ liệu thô vì notebook xuất bản công khai không nên chứa nguyên văn bài báo. Các thống kê dưới đây được tính trước mọi thao tác làm sạch.

In [ ]:
def load_csv(filename, label):
    path = DATA_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f'Cần file CSV, không phải thư mục: {path}')
    frame = pd.read_csv(path, low_memory=False)
    required = {'title', 'text', 'subject', 'date'}
    if missing := required.difference(frame.columns):
        raise ValueError(f'{path.name} thiếu cột: {sorted(missing)}')
    frame = frame[['title', 'text', 'subject', 'date']].copy()
    frame['label'] = label
    return frame

raw = pd.concat([load_csv('Fake.csv', 'FAKE'), load_csv('True.csv', 'REAL')], ignore_index=True)
missing_by_label = raw.groupby('label')[['title', 'text', 'subject', 'date']].agg(
    lambda column: int(column.fillna('').astype(str).str.strip().eq('').sum())
)
print('Số bài theo nhãn:')
display(raw.groupby('label').size().rename('so_bai').to_frame())
print('Ô trống/NA theo nhãn:')
display(missing_by_label)
exact_dupes = int(raw.duplicated(subset=['title', 'text'], keep=False).sum())
conflict_groups = int(raw.groupby(['title', 'text'], dropna=False)['label'].nunique().gt(1).sum())
print(f'Bài nằm trong nhóm trùng chính xác (title + text): {exact_dupes:,}')
print(f'Nhóm trùng chính xác nhưng có cả hai nhãn: {conflict_groups:,}')

data = raw.fillna({'title': '', 'text': '', 'subject': '', 'date': ''}).copy()
for column in ['title', 'text', 'subject', 'date']:
    data[column] = data[column].astype(str)
if data['label'].nunique() != 2:
    raise ValueError('Cần đủ hai nhãn để so sánh.')

## 2. Độ dài tiêu đề và nội dung

Bảng dùng toàn bộ phân phối. Biểu đồ giới hạn trục X ở bách phân vị 99 để các bài cực dài không che phần còn lại; số liệu gốc không bị loại.

In [ ]:
for field in ['title', 'text']:
    data[f'{field}_words'] = data[field].str.count(r"\b[\w']+\b")
    data[f'{field}_chars'] = data[field].str.len()
length_cols = ['title_words', 'title_chars', 'text_words', 'text_chars']
length_summary = data.groupby('label')[length_cols].agg(['mean', 'median', 'std'])
display(length_summary.round(1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for axis, field, title in zip(axes, ['title_words', 'text_words'], ['Tiêu đề (từ)', 'Nội dung (từ)']):
    cap = max(1, float(data[field].quantile(0.99)))
    for label in ['FAKE', 'REAL']:
        values = data.loc[data.label == label, field].clip(upper=cap)
        axis.hist(values, bins=35, alpha=0.55, density=True, color=COLORS[label], label=label)
    axis.set(title=title, xlabel='Số từ (giới hạn ở P99)', ylabel='Mật độ')
    axis.legend()
save_plot('01_length.png')

## 3. Phân bố chủ đề và tín hiệu rò rỉ

Cột `subject` có thể phản ánh quy trình thu thập từng lớp hơn là chủ đề chung. Chỉ dùng nó để **kiểm toán dữ liệu**, không đưa thẳng vào mô hình hay chia fold theo chủ đề trước khi issue #2/#4 kiểm tra.

In [ ]:
subjects = pd.crosstab(data['subject'].replace('', '(trống)'), data['label']).reindex(columns=['FAKE', 'REAL'], fill_value=0)
subjects['tong'] = subjects.sum(axis=1)
subjects['ty_le_fake'] = (subjects['FAKE'] / subjects['tong']).round(3)
display(subjects.sort_values('tong', ascending=False))
exclusive = subjects[(subjects['FAKE'] == 0) | (subjects['REAL'] == 0)]
print(f'Chủ đề chỉ xuất hiện trong một nhãn: {len(exclusive)}/{len(subjects)}')
print(f'Tỷ lệ bài thuộc chủ đề một nhãn: {exclusive.tong.sum() / len(data):.1%}')

subjects[['FAKE', 'REAL']].sort_values('FAKE', ascending=True).plot(
    kind='barh', stacked=True, color=[COLORS['FAKE'], COLORS['REAL']], figsize=(10, max(4, len(subjects) * 0.42))
)
plt.title('Số bài theo subject và nhãn'); plt.xlabel('Số bài'); plt.ylabel('subject')
save_plot('02_subject.png')

prefix = data['text'].str.slice(0, 220)
data['reuters_marker'] = prefix.str.contains(r'\(\s*Reuters\s*\)\s*[-–—]', case=False, regex=True)
data['reuters_early'] = prefix.str.contains(r'\bReuters\b', case=False, regex=True)
source_summary = data.groupby('label')[['reuters_marker', 'reuters_early']].agg(['sum', 'mean'])
display(source_summary.round(3))
print('Các tỷ lệ này chỉ là tín hiệu cần kiểm toán thêm: regex có thể bỏ sót hoặc nhận nhầm định dạng nguồn.')

## 4. Từ vựng đặc trưng theo nhãn

Chỉ báo dưới đây là **chênh lệch tần suất xuất hiện trong tài liệu**, không phải trọng số của mô hình hay bằng chứng đúng/sai. Vectorizer được fit trên toàn bộ dữ liệu chỉ để mô tả EDA; tuyệt đối không tái sử dụng nó trong train/test. Khi huấn luyện, fit vectorizer chỉ trên train. Tên nguồn hoặc chủ đề hiện diện trong top từ là lý do phải kiểm toán rò rỉ.

In [ ]:
documents = (data['title'] + ' ' + data['text']).tolist()
vectorizer = CountVectorizer(stop_words='english', binary=True, min_df=2 if len(data) < 100 else 10, max_features=15000)
try:
    matrix = vectorizer.fit_transform(documents)
except ValueError as exc:
    raise ValueError('Không đủ từ chung giữa các bài để tính từ vựng; kiểm tra CSV có văn bản hay không.') from exc
terms = vectorizer.get_feature_names_out()
fake_mask = data.label.eq('FAKE').to_numpy()
fake_share = np.asarray(matrix[fake_mask].sum(axis=0)).ravel() / fake_mask.sum()
real_share = np.asarray(matrix[~fake_mask].sum(axis=0)).ravel() / (~fake_mask).sum()
term_stats = pd.DataFrame({'term': terms, 'fake_share': fake_share, 'real_share': real_share})
term_stats['difference'] = term_stats.fake_share - term_stats.real_share
fake_terms = term_stats.nlargest(15, 'difference').copy()
real_terms = term_stats.nsmallest(15, 'difference').copy()
print('15 từ thiên về FAKE theo chênh lệch tỷ lệ bài chứa từ:')
display(fake_terms.round(3).reset_index(drop=True))
print('15 từ thiên về REAL theo cùng cách tính:')
display(real_terms.round(3).reset_index(drop=True))

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for axis, table, label in zip(axes, [fake_terms, real_terms], ['FAKE', 'REAL']):
    axis.barh(table.term.iloc[::-1], table.difference.abs().iloc[::-1], color=COLORS[label])
    axis.set(title=f'Từ thiên về {label}', xlabel='Chênh lệch tỷ lệ tài liệu (tuyệt đối)')
save_plot('03_terms.png')

## 5. Phong cách viết (proxy đơn giản)

So sánh độ dài câu ước lượng, dấu câu trên 1000 ký tự, dấu chấm than/hỏi và tỷ lệ chữ hoa. Đây là đặc điểm bề mặt: độ dài, nguồn tin, chủ đề và lỗi tách câu đều có thể tác động đến chúng.

In [ ]:
body = data['text']
data['words_per_sentence'] = data.text_words / body.str.count(r'[.!?]+').add(1)
data['punct_per_1000'] = body.str.count(r'[^\w\s]') * 1000 / data.text_chars.clip(lower=1)
data['exclaim_question_per_1000'] = body.str.count(r'[!?]') * 1000 / data.text_chars.clip(lower=1)
data['uppercase_share'] = body.str.count(r'[A-Z]') / body.str.count(r'[A-Za-z]').clip(lower=1)
style_cols = ['words_per_sentence', 'punct_per_1000', 'exclaim_question_per_1000', 'uppercase_share']
display(data.groupby('label')[style_cols].agg(['mean', 'median']).round(3))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for axis, column in zip(axes.flat, style_cols):
    clipped = data[column].clip(upper=data[column].quantile(0.99))
    axis.boxplot([clipped[data.label.eq(label)] for label in ['FAKE', 'REAL']], tick_labels=['FAKE', 'REAL'], showfliers=False)
    axis.set(title=column, ylabel='Giá trị; hiển thị tối đa P99')
save_plot('04_style.png')

## 6. Tóm tắt để chuyển giao DATA-02

Chỉ báo định lượng bên dưới lấy từ lần chạy hiện tại; cần kiểm tra lại bằng quan sát và phương pháp ở issue #2. Không coi khác biệt thống kê là quan hệ nhân quả hay thước đo tính thật/giả.

In [ ]:
median_words = data.groupby('label')['text_words'].median()
reuters_rates = data.groupby('label')['reuters_marker'].mean()
print(f'Số bài: {len(data):,} | FAKE: {fake_mask.sum():,} | REAL: {(~fake_mask).sum():,}')
print(f'Trung vị số từ nội dung: FAKE {median_words["FAKE"]:.0f}, REAL {median_words["REAL"]:.0f}')
print(f'Subject chỉ có một nhãn: {len(exclusive)}/{len(subjects)} nhóm, chiếm {exclusive.tong.sum() / len(data):.1%} số bài')
print(f'Dấu (Reuters) ở đầu bài: FAKE {reuters_rates["FAKE"]:.1%}, REAL {reuters_rates["REAL"]:.1%}')
print('Chuyển DATA-02: rà thêm dateline, tên hãng tin, định dạng mở đầu, bài trùng/gần trùng và ablation trước/sau làm sạch.')
print('Giới hạn: subject/date có thể chứa dấu nguồn; mẫu chỉ là báo tiếng Anh; từ đặc trưng có thể phản ánh nguồn hoặc chủ đề.')